# Global Optimization Script

This script finds the optimal parameter values of the model that best meet **8 calibration targets and 1 soft target** suggested by the literature using an optimization algorithm called **Differential Evolution**. The parameter values (12 in number) obtained here are exported as a `JSON` file, namely, `optimal_parameters.json` These are the same values that are used in `60_min_optimized_script.ipynb` for model simulation run and generation of results. So, the correct workflow for this project is to run this script first to get the optimal parameter values, and import these values to the simulation script, `60_min_optimized_script.ipynb`, (done manually) to get the simulation results (in the `/plots` directory).

The first part of this script is shared by the simulation script as well, as it contains the core model written in Antimony and associated functions for the model to reach steady state. The second part of this script deals solely with the optimization problem. 

# Model Definition and Specification

This part of the script, shared by `60_min_optimized_script.ipynb`, deals with defining the model in Antimony and specifying functions for it to reach steady state and other important experimental variables.

## Import Statements

In [1]:
import tellurium as te
import roadrunner
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.optimize import differential_evolution
import json

## Antimony String (Model)

ODE model written in Antimony.

In [2]:
# Antimony model
antimony_model = """
model Ecoli_Cfreundii_AmpC_Validation()
 
    // =====================================================================
    // 0. INITIAL CONDITIONS  (all pools start empty; burn-in fills them)
    // =====================================================================
    UDP_NAG = 0.0; UDP_NAG_ep = 0.0; UDP_NAM = 0.0; UDP_NAM_Ala = 0.0
    UDP_NAM_Ala_Glu = 0.0; UDP_NAM_tri_DAP = 0.0; UDP_NAM_penta = 0.0
    Lipid_I_cyto = 0.0; Lipid_II_cyto = 0.0; Lipid_II_peri = 0.0
    PG_UP = 0.0; PG_UT = 0.0; PG_XL = 0.0
    Disacch_P_peri = 0.0; Disacch_T_peri = 0.0
    Disacch_P_cyto = 0.0; Disacch_T_cyto = 0.0; Disacch_Tri = 0.0; Disacch = 0.0
    Anh_P = 0.0; Anh_T = 0.0; Anh_Tri = 0.0
    anhNAM = 0.0; NAM_6_P = 0.0; NAG_6_P = 0.0
    GlcN_6_P = 0.0; GlcN_1_P = 0.0; NAG = 0.0
    Stem_Penta = 0.0; Stem_Tetra = 0.0; Stem_Tri = 0.0
    mRNA_AmpR = 0.0; AmpR = 0.0; AmpR_total = 0.0; mRNA_AmpC = 0.0; AmpC = 0.0
    FtsZ = 0.0; BL = 0.0
    Und_PP_peri = 0.0; Und_P_peri = 0.0; Und_P_cyto = 0.0
    PBP1a = 0.0; PBP1b = 0.0; PBP2 = 0.0; PBP3 = 0.0
    PBP4 = 0.0; PBP5 = 0.0; PBP6 = 0.0; PBP7 = 0.0
    
 
    // =====================================================================
    // 1. GLOBAL / CELL GEOMETRY
    // =====================================================================
    mu = 5.78e-04                  // E. coli LB, ~20 min doubling
    V_cyto = 0.8; V_peri = 0.2
    v_in_NAG   = 6.00e-03
    v_de_novo  = 2.03814e-03
 
    // =====================================================================
    // 2. MUR PATHWAY (all E. coli)
    // =====================================================================
    V_max_MurA = 4.19e-03; K_m_MurA = 0.015; K_inhibit = 0.0009; n_MurA = 1
    V_max_MurB = 3.80e-03; K_m_MurB = 0.015
    V_max_MurC = 8.01e-03; K_m_MurC = 0.04
    V_max_MurD = 7.10e-03; K_m_MurD = 0.0073
    V_max_MurE = 2.86e-03; K_m_MurE = 0.076
    V_max_MurF = 5.47e-03; K_m_MurF = 0.078; K_I_MurF = 0.008 // 1 mM originally
 
    V_max_MraY = 9.23360e-04; K_M_UDP_NAM_pentapeptide = 0.0056; K_M_UndP = 2.00000e-02
    V_max_MurG = 6.68858e-04; K_m_LipidI = 0.02; K_m_UDP_NAG = 0.058
    V_max_MurJ = 2.22e-03; K_M_MurJ = 0.002
 
    // =====================================================================
    // 3. PERIPLASMIC SYNTHASES
    // =====================================================================
    V_max_RodA = 1.03920e-03; K_M_RodA = 0.002
    V_max_FtsW = 1.69e-03; K_M_FtsW = 0.002
 
    k_cat_PBP1a_GT = 5.00000e-01;  K_M_PBP1a_GT = 0.002
    k_cat_PBP1b_GT = 100;  K_M_PBP1b_GT = 0.002
    k_cat_PBP1a_TP = 3.86; K_M_PBP1a_TP = 0.5 // k_cat_PBP1a_TP = 3.86
    k_cat_PBP1b_TP = 3.86; K_M_PBP1b_TP = 0.5
    k_cat_PBP2 = 3.86; K_M_PBP2 = 0.5 // k_cat_PBP2 = 3.86
    k_cat_PBP3 = 3.86; K_M_PBP3 = 0.5
    K_M_acc = 0.5                              // acceptor saturation
 
    // LMW PBPs: carboxypeptidase / endopeptidase
    k_cat_PBP4_CP = 3.47; K_M_PBP4_CP = 0.0204
    k_cat_PBP4_EP = 3.47; K_M_PBP4_EP = 0.0204
    k_cat_PBP5 = 6.56750e-01; K_M_PBP5 = 0.0204
    k_cat_PBP6 = 6.06594e-02; K_M_PBP6 = 0.0204
    k_cat_PBP7 = 3.47; K_M_PBP7 = 0.0204
 
    // =====================================================================
    // 4. HYDROLASES  (Slt = lumped LT; V_max from 40-45%/generation turnover)
    // =====================================================================
    V_max_Slt = 5.00000e-02; K_M_Slt = 1.8
    V_max_Ami_UP = 3.00e-04; K_M_Ami_UP = 1.5
    V_max_Ami_UT = 3.00e-04; K_M_Ami_UT = 1.5
    V_max_Ami_XL = 3.00e-04; K_M_Ami_XL = 1.5
 
    // =====================================================================
    // 5. TRANSPORT AND CYTOPLASMIC RECYCLING
    // =====================================================================
    V_max_AmpG = 1.98e-03; K_M_AmpG = 0.01
 
    V_max_NagZ_Disacch_P   = 1.22e-03; K_M_NagZ_Disacch_P   = 0.104
    V_max_NagZ_Disacch_T   = 1.04e-03; K_M_NagZ_Disacch_T   = 0.095
    V_max_NagZ_Disacch_Tri = 8.61e-04; K_M_NagZ_Disacch_Tri = 0.086
    V_max_NagZ_Disacch     = 3.15e-04; K_M_NagZ_Disacch     = 0.059
 
    V_max_LdcA           = 3.10e-03; K_M_LdcA           = 0.17   // on Anh_T
    V_max_LdcA_Disacch_T = 1.89e-03; K_M_LdcA_Disacch_T = 0.17
    V_max_LdcA_stem      = 1.89e-03; K_M_LdcA_stem      = 0.17   // on free Stem_Tetra
 
    V_max_AmpD_P   = 4.64e-04; K_M_AmpD_P   = 0.5
    V_max_AmpD_T   = 7.59e-04; K_M_AmpD_T   = 0.43
    V_max_AmpD_Tri = 1.05e-03; K_M_AmpD_Tri = 0.36
    V_max_AmpD_Disacch_P   = 1.69e-05; K_M_AmpD_Disacch_P   = 1.76
    V_max_AmpD_Disacch_T   = 2.76e-05; K_M_AmpD_Disacch_T   = 1.514
    V_max_AmpD_Disacch_Tri = 3.83e-05; K_M_AmpD_Disacch_Tri = 1.267
 
    V_max_AnmK = 7.82e-03; K_M_AnmK = 0.5
 
    // Mpl shortcut (E. coli, sourced)
    V_max_Mpl = 3.49e-03; K_M_Mpl_NAM = 0.25
    K_M_Mpl_Tri = 0.1; K_M_Mpl_Penta = 0.1
 
    // Amino-sugar re-entry (E. coli MurQ route; AmgK/MurU/MupP absent)
    V_max_MurQ = 1.68e-03; K_M_MurQ = 0.2
    V_max_NagA = 1.66e-02; K_M_NagA = 0.5
    V_max_NagB = 1.99e-01; K_M_NagB = 0.5
    V_max_NagK = 1.32e-02; K_M_NagK = 0.127
    V_max_GlmS = 5.31e-02; K_M_GlmS_F6P = 2.0; K_M_GlmS_Gln = 0.4
    V_max_GlmM_fwd = 6.21e-02; K_M_GlmM_6P = 0.2
    V_max_GlmM_rev = 6.21e-02; K_M_GlmM_1P = 0.2
    V_max_GlmU = 1.69e-02; K_M_GlmU = 0.1
 
    // Fixed buffered boundary conditions
    Fru_6_P_basal = 0.6
    Gln_basal = 4.0
 
    // =====================================================================
    // 6. UND-P RECYCLING
    // =====================================================================
    V_max_BacA = 4.12609e-03; K_M_BacA = 0.53
    V_max_DedA = 1.25238e-02; K_M_DedA = 0.002
 
    // =====================================================================
    // 7. AmpR / AmpC  (C. freundii genes on pNU305)
    // =====================================================================
    N_copy = 17
    K_R = 0.2; K_A1 = 0.32; K_A2 = 3.2; n_AmpR = 4
    k_tx_rep = 1.17e-06; k_tx_basal = 2.93e-05; k_tx_act = 1.17e-03
    v_tx_AmpR = 6.25e-10; K_autoR = 3.32e-04; n_autoR = 2
    k_tl_AmpR = 0.16;  k_deg_mRNA_AmpR = 3.85e-03; k_deg_p_AmpR = 1.00e-06
    k_tl = 0.16;       k_deg_mRNA_AmpC = 3.85e-03; k_deg_p_AmpC = 1.00e-06
 
    // =====================================================================
    // 8. FtsZ (ftsZ84 temperature-sensitive allele)
    // =====================================================================
    v_synth_FtsZ = 7.04e-06; k_deg_p_FtsZ = 2.70e-04
    K_Z = 0.001; n_Z = 4
 
    // =====================================================================
    // 9. CEFOXITIN
    // =====================================================================
    k_diff_BL = 8.16                  // WEAK estimate - sweep this
    k_cat_AmpC = 0.23; K_M_AmpC = 7.0e-05
    V_max_eff = 0.0; K_M_eff = 0.001  // cefoxitin not an AcrAB substrate
 
    k_acyl_PBP1a = 0.183
    k_acyl_PBP1b = 2.35
    k_acyl_PBP2  = 0.00016
    k_acyl_PBP3  = 0.274
    k_acyl_PBP4  = 16.5
    k_acyl_PBP5  = 2.35
    k_acyl_PBP6  = 2.35
    k_acyl_PBP7  = 16.5
 
    v_synth_PBP1a = 8.90e-08
    v_synth_PBP1b = 4.89e-07
    v_synth_PBP2  = 4.96e-08
    v_synth_PBP3  = 9.36e-08
    v_synth_PBP4  = 5.61e-08
    v_synth_PBP5  = 3.84e-06
    v_synth_PBP6  = 5.24e-06
    v_synth_PBP7  = 7.40e-08
 
    PBP2_ss = 8.59e-05
    PBP3_ss = 1.62e-04

    f_PBP1b_septal = 0.05
 
    // =====================================================================
    // 10. EXPERIMENTAL CONTROLS
    // =====================================================================
    ts_allele = 0     // 0 = WT ftsZ+ ; 1 = ftsZ84
    drug_on   = 0     // 0 = no cefoxitin ; 1 = cefoxitin
    dose      = 0.0187
    t_shift   = 0     // 30->42 C temperature shift
    t_drug    = 1800  // cefoxitin added 30 min AFTER shift
 
    // =====================================================================
    // 11. ALGEBRAIC RULES
    // =====================================================================
    Theta  := 1 - ts_allele * piecewise(1, time >= t_shift, 0)
    BL_ext := drug_on * dose * piecewise(1, time >= t_drug, 0)
 
    Z := Theta * (FtsZ^n_Z) / (K_Z^n_Z + FtsZ^n_Z)
 
    A_acc := (PG_UP + PG_UT) / (K_M_acc + PG_UP + PG_UT)

    XL_f := PG_XL / (PG_UP + PG_UT + PG_XL)
 
    J_TP := A_acc * ( (k_cat_PBP2*PBP2*PG_UP)/(K_M_PBP2 + PG_UP) + Z*(k_cat_PBP3*PBP3*PG_UP)/(K_M_PBP3 + PG_UP) + (k_cat_PBP1a_TP*PBP1a*PG_UP)/(K_M_PBP1a_TP + PG_UP) + Z*(k_cat_PBP1b_TP*PBP1b*PG_UP)/(K_M_PBP1b_TP + PG_UP) )
 
    J_endo := (k_cat_PBP4_EP*PBP4*PG_XL)/(K_M_PBP4_EP + PG_XL) + (k_cat_PBP7*PBP7*PG_XL)/(K_M_PBP7 + PG_XL)
 
    J_RodA   := (PBP2/PBP2_ss) * (V_max_RodA*Lipid_II_peri)/(K_M_RodA + Lipid_II_peri)
    J_FtsW   := Z * (PBP3/PBP3_ss) * (V_max_FtsW*Lipid_II_peri)/(K_M_FtsW + Lipid_II_peri)
    J_PBP1aG := (k_cat_PBP1a_GT*PBP1a*Lipid_II_peri)/(K_M_PBP1a_GT + Lipid_II_peri)
    J_PBP1bG := Z * (k_cat_PBP1b_GT*PBP1b*Lipid_II_peri)/(K_M_PBP1b_GT + Lipid_II_peri)
    J_poly   := J_RodA + J_FtsW + J_PBP1aG + J_PBP1bG
 
    J_MraY := V_max_MraY * (UDP_NAM_penta/(K_M_UDP_NAM_pentapeptide + UDP_NAM_penta)) * (Und_P_cyto/(K_M_UndP + Und_P_cyto))
    J_MurG := (V_max_MurG*Lipid_I_cyto*UDP_NAG)/((K_m_LipidI + Lipid_I_cyto)*(K_m_UDP_NAG + UDP_NAG))
    J_MurA := (V_max_MurA*UDP_NAG)/(K_m_MurA + UDP_NAG) * (1/(1 + (UDP_NAM/K_inhibit)^n_MurA))
    J_MurF := (V_max_MurF*UDP_NAM_tri_DAP)/(K_m_MurF + UDP_NAM_tri_DAP + (UDP_NAM_tri_DAP^2)/K_I_MurF)
    J_MurJ := (V_max_MurJ*Lipid_II_cyto)/(K_M_MurJ + Lipid_II_cyto)
 
    J_Mpl_Tri   := V_max_Mpl * (UDP_NAM/(K_M_Mpl_NAM + UDP_NAM)) * (Stem_Tri/(K_M_Mpl_Tri + Stem_Tri))
    J_Mpl_Penta := V_max_Mpl * (UDP_NAM/(K_M_Mpl_NAM + UDP_NAM)) * (Stem_Penta/(K_M_Mpl_Penta + Stem_Penta))
 
    J_GlmS := V_max_GlmS * (Fru_6_P_basal/(K_M_GlmS_F6P + Fru_6_P_basal)) * (Gln_basal/(K_M_GlmS_Gln + Gln_basal))
    J_GlmM := (V_max_GlmM_fwd*(GlcN_6_P/K_M_GlmM_6P) - V_max_GlmM_rev*(GlcN_1_P/K_M_GlmM_1P)) / (1 + GlcN_6_P/K_M_GlmM_6P + GlcN_1_P/K_M_GlmM_1P)
 
    J_Slt_UP := (V_max_Slt*PG_UP)/(K_M_Slt + PG_UP)
    J_Slt_UT := (V_max_Slt*PG_UT)/(K_M_Slt + PG_UT)
    J_AmpG_P := (V_max_AmpG*Disacch_P_peri)/(K_M_AmpG + Disacch_P_peri)
    J_AmpG_T := (V_max_AmpG*Disacch_T_peri)/(K_M_AmpG + Disacch_T_peri)
 
    J_NagZ_P   := (V_max_NagZ_Disacch_P*Disacch_P_cyto)/(K_M_NagZ_Disacch_P + Disacch_P_cyto)
    J_NagZ_T   := (V_max_NagZ_Disacch_T*Disacch_T_cyto)/(K_M_NagZ_Disacch_T + Disacch_T_cyto)
    J_NagZ_Tri := (V_max_NagZ_Disacch_Tri*Disacch_Tri)/(K_M_NagZ_Disacch_Tri + Disacch_Tri)
    J_NagZ_D   := (V_max_NagZ_Disacch*Disacch)/(K_M_NagZ_Disacch + Disacch)
 
    J_AmpD_P   := (V_max_AmpD_P*Anh_P)/(K_M_AmpD_P + Anh_P)
    J_AmpD_T   := (V_max_AmpD_T*Anh_T)/(K_M_AmpD_T + Anh_T)
    J_AmpD_Tri := (V_max_AmpD_Tri*Anh_Tri)/(K_M_AmpD_Tri + Anh_Tri)
    J_AmpD_DP  := (V_max_AmpD_Disacch_P*Disacch_P_cyto)/(K_M_AmpD_Disacch_P + Disacch_P_cyto)
    J_AmpD_DT  := (V_max_AmpD_Disacch_T*Disacch_T_cyto)/(K_M_AmpD_Disacch_T + Disacch_T_cyto)
    J_AmpD_DTri := (V_max_AmpD_Disacch_Tri*Disacch_Tri)/(K_M_AmpD_Disacch_Tri + Disacch_Tri)
 
    J_LdcA_AnhT := (V_max_LdcA*Anh_T)/(K_M_LdcA + Anh_T)
    J_LdcA_DT   := (V_max_LdcA_Disacch_T*Disacch_T_cyto)/(K_M_LdcA_Disacch_T + Disacch_T_cyto)
    J_LdcA_stem := (V_max_LdcA_stem*Stem_Tetra)/(K_M_LdcA_stem + Stem_Tetra)
 
    J_BacA := (V_max_BacA*Und_PP_peri)/(K_M_BacA + Und_PP_peri)
    J_DedA := (V_max_DedA*Und_P_peri)/(K_M_DedA + Und_P_peri)

        // --- newly named terms, so every equation's pieces are individually inspectable ---
    J_MurB := (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep)
    J_MurC := (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM)
    J_MurD := (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala)
    J_MurE := (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu)
    J_GlmU := (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P)

    J_PBP5_CP := (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP)
    J_PBP6_CP := (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP)
    J_PBP4_CP := (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP)
    J_Ami_UP  := (V_max_Ami_UP*PG_UP)/(K_M_Ami_UP + PG_UP)
    J_Ami_UT  := (V_max_Ami_UT*PG_UT)/(K_M_Ami_UT + PG_UT)
    J_Ami_XL  := (V_max_Ami_XL*PG_XL)/(K_M_Ami_XL + PG_XL)

    J_AnmK := (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM)
    J_MurQ := (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P)
    J_NagK := (V_max_NagK*NAG)/(K_M_NagK + NAG)
    J_NagA := (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P)
    J_NagB := (V_max_NagB*GlcN_6_P)/(K_M_NagB + GlcN_6_P)

    J_tx_AmpR := N_copy*v_tx_AmpR*(1/(1 + (AmpR_total/K_autoR)^n_autoR))
    J_tl_AmpR := k_tl_AmpR*mRNA_AmpR

    J_tx_AmpC_rep   := N_copy*k_tx_rep*(UDP_NAM_penta/K_R)^n_AmpR*AmpR
    J_tx_AmpC_basal := N_copy*k_tx_basal*AmpR
    J_tx_AmpC_act   := N_copy*k_tx_act*((Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR)*AmpR
    J_tl_AmpC := (V_cyto/V_peri)*k_tl*mRNA_AmpC

    J_diff_BL := k_diff_BL*(BL_ext - BL)
    J_AmpC_hydrolysis := (k_cat_AmpC*AmpC*BL)/(K_M_AmpC + BL)
    J_eff := (V_max_eff*BL)/(K_M_eff + BL)
    J_acyl_PBP1a := k_acyl_PBP1a*PBP1a*BL
    J_acyl_PBP1b := k_acyl_PBP1b*PBP1b*BL
    J_acyl_PBP2  := k_acyl_PBP2*PBP2*BL
    J_acyl_PBP3  := k_acyl_PBP3*PBP3*BL
    J_acyl_PBP4  := k_acyl_PBP4*PBP4*BL
    J_acyl_PBP5  := k_acyl_PBP5*PBP5*BL
    J_acyl_PBP6  := k_acyl_PBP6*PBP6*BL
    J_acyl_PBP7  := k_acyl_PBP7*PBP7*BL

    XL_f := PG_XL / (PG_UP + PG_UT + PG_XL)

    denom_AmpR := 1 + (UDP_NAM_penta/K_R)^n_AmpR + (Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR
    AmpR := AmpR_total / denom_AmpR
 
    // =====================================================================
    // 12. CYTOPLASMIC PRECURSOR BIOSYNTHESIS
    // =====================================================================
    UDP_NAG' = v_in_NAG + (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P) - J_MurA - J_MurG - mu*UDP_NAG
    UDP_NAG_ep' = J_MurA - (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep) - mu*UDP_NAG_ep
    UDP_NAM' = (V_max_MurB*UDP_NAG_ep)/(K_m_MurB + UDP_NAG_ep) - (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM) - J_Mpl_Penta - J_Mpl_Tri - mu*UDP_NAM
    UDP_NAM_Ala' = (V_max_MurC*UDP_NAM)/(K_m_MurC + UDP_NAM) - (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala) - mu*UDP_NAM_Ala
    UDP_NAM_Ala_Glu' = (V_max_MurD*UDP_NAM_Ala)/(K_m_MurD + UDP_NAM_Ala) - (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu) - mu*UDP_NAM_Ala_Glu
    UDP_NAM_tri_DAP' = (V_max_MurE*UDP_NAM_Ala_Glu)/(K_m_MurE + UDP_NAM_Ala_Glu) + J_Mpl_Tri - J_MurF - mu*UDP_NAM_tri_DAP
    UDP_NAM_penta' = J_MurF + J_Mpl_Penta - J_MraY - mu*UDP_NAM_penta
    Lipid_I_cyto' = J_MraY - J_MurG - mu*Lipid_I_cyto
    Lipid_II_cyto' = J_MurG - J_MurJ - mu*Lipid_II_cyto
 
    // =====================================================================
    // 13. PERIPLASMIC REACTIONS
    // =====================================================================
    Lipid_II_peri' = (V_cyto/V_peri)*J_MurJ - J_poly - mu*Lipid_II_peri
 
    PG_UP' = J_poly - J_TP - (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP) - (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP) - (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP) - J_Slt_UP - (V_max_Ami_UP*PG_UP)/(K_M_Ami_UP + PG_UP) - mu*PG_UP
 
    PG_UT' = (k_cat_PBP5*PBP5*PG_UP)/(K_M_PBP5 + PG_UP) + (k_cat_PBP6*PBP6*PG_UP)/(K_M_PBP6 + PG_UP) + (k_cat_PBP4_CP*PBP4*PG_UP)/(K_M_PBP4_CP + PG_UP) + J_endo - J_Slt_UT - (V_max_Ami_UT*PG_UT)/(K_M_Ami_UT + PG_UT) - mu*PG_UT
 
    PG_XL' = J_TP - J_endo - (V_max_Ami_XL*PG_XL)/(K_M_Ami_XL + PG_XL) - mu*PG_XL
 
    Disacch_P_peri' = J_Slt_UP - J_AmpG_P - mu*Disacch_P_peri
    Disacch_T_peri' = J_Slt_UT - J_AmpG_T - mu*Disacch_T_peri
 
    // =====================================================================
    // 14. CYTOPLASMIC RECYCLING
    // =====================================================================
    Disacch_P_cyto' = (V_peri/V_cyto)*J_AmpG_P - J_NagZ_P - J_AmpD_DP - mu*Disacch_P_cyto
    Disacch_T_cyto' = (V_peri/V_cyto)*J_AmpG_T - J_NagZ_T - J_AmpD_DT - J_LdcA_DT - mu*Disacch_T_cyto
    Disacch_Tri' = J_LdcA_DT - J_NagZ_Tri - J_AmpD_DTri - mu*Disacch_Tri
    Disacch' = J_AmpD_DP + J_AmpD_DT + J_AmpD_DTri - J_NagZ_D - mu*Disacch
 
    Anh_P' = J_NagZ_P - J_AmpD_P - mu*Anh_P
    Anh_T' = J_NagZ_T - J_AmpD_T - J_LdcA_AnhT - mu*Anh_T
    Anh_Tri' = J_NagZ_Tri + J_LdcA_AnhT - J_AmpD_Tri - mu*Anh_Tri
 
    anhNAM' = J_AmpD_P + J_AmpD_T + J_AmpD_Tri + J_NagZ_D - (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM) - mu*anhNAM
    NAM_6_P' = (V_max_AnmK*anhNAM)/(K_M_AnmK + anhNAM) - (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P) - mu*NAM_6_P
    NAG_6_P' = (V_max_MurQ*NAM_6_P)/(K_M_MurQ + NAM_6_P) + (V_max_NagK*NAG)/(K_M_NagK + NAG) - (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P) - mu*NAG_6_P
    GlcN_6_P' = (V_max_NagA*NAG_6_P)/(K_M_NagA + NAG_6_P) + J_GlmS - (V_max_NagB*GlcN_6_P)/(K_M_NagB + GlcN_6_P) - J_GlmM - mu*GlcN_6_P
    GlcN_1_P' = J_GlmM - (V_max_GlmU*GlcN_1_P)/(K_M_GlmU + GlcN_1_P) - mu*GlcN_1_P
    NAG' = J_NagZ_D + J_NagZ_P + J_NagZ_T + J_NagZ_Tri - (V_max_NagK*NAG)/(K_M_NagK + NAG) - mu*NAG
 
    Stem_Penta' = J_AmpD_DP + J_AmpD_P - J_Mpl_Penta - mu*Stem_Penta
    Stem_Tetra' = J_AmpD_DT + J_AmpD_T - J_LdcA_stem - mu*Stem_Tetra
    Stem_Tri' = J_AmpD_DTri + J_AmpD_Tri + J_LdcA_stem - J_Mpl_Tri - mu*Stem_Tri
 
    // =====================================================================
    // 15. AmpR / AmpC EXPRESSION
    // =====================================================================
    mRNA_AmpR' = N_copy*v_tx_AmpR*(1/(1 + (AmpR_total/K_autoR)^n_autoR)) - (k_deg_mRNA_AmpR + mu)*mRNA_AmpR
    AmpR_total' = k_tl_AmpR*mRNA_AmpR - (k_deg_p_AmpR + mu)*AmpR_total
 
    mRNA_AmpC' = N_copy*( k_tx_rep*(UDP_NAM_penta/K_R)^n_AmpR + k_tx_basal + k_tx_act*( (Anh_P/K_A1)^n_AmpR + (Disacch_P_cyto/K_A2)^n_AmpR ) )*AmpR - (k_deg_mRNA_AmpC + mu)*mRNA_AmpC
    AmpC' = (V_cyto/V_peri)*k_tl*mRNA_AmpC - (k_deg_p_AmpC + mu)*AmpC
 
    // =====================================================================
    // 16. FtsZ
    // =====================================================================
    FtsZ' = v_synth_FtsZ - (k_deg_p_FtsZ + mu)*FtsZ
 
    // =====================================================================
    // 17. CEFOXITIN AND PBP ACYLATION
    // =====================================================================
    BL' = k_diff_BL*(BL_ext - BL) - (k_cat_AmpC*AmpC*BL)/(K_M_AmpC + BL) - (V_max_eff*BL)/(K_M_eff + BL) - (k_acyl_PBP1a*PBP1a + k_acyl_PBP1b*PBP1b + k_acyl_PBP2*PBP2 + k_acyl_PBP3*PBP3 + k_acyl_PBP4*PBP4 + k_acyl_PBP5*PBP5 + k_acyl_PBP6*PBP6 + k_acyl_PBP7*PBP7)*BL - mu*BL
 
    PBP1a' = v_synth_PBP1a - k_acyl_PBP1a*PBP1a*BL - mu*PBP1a
    PBP1b' = v_synth_PBP1b - k_acyl_PBP1b*PBP1b*BL - mu*PBP1b
    PBP2'  = v_synth_PBP2  - k_acyl_PBP2*PBP2*BL   - mu*PBP2
    PBP3'  = v_synth_PBP3  - k_acyl_PBP3*PBP3*BL   - mu*PBP3
    PBP4'  = v_synth_PBP4  - k_acyl_PBP4*PBP4*BL   - mu*PBP4
    PBP5'  = v_synth_PBP5  - k_acyl_PBP5*PBP5*BL   - mu*PBP5
    PBP6'  = v_synth_PBP6  - k_acyl_PBP6*PBP6*BL   - mu*PBP6
    PBP7'  = v_synth_PBP7  - k_acyl_PBP7*PBP7*BL   - mu*PBP7
 
    // =====================================================================
    // 18. UND-P RECYCLING
    // =====================================================================
    Und_PP_peri' = v_de_novo + J_poly - J_BacA - mu*Und_PP_peri
    Und_P_peri'  = J_BacA - J_DedA - mu*Und_P_peri
    Und_P_cyto'  = (V_peri/V_cyto)*J_DedA - J_MraY - mu*Und_P_cyto
 
end
""" 

## Steady-State Helper Functions

### Solver Parameters

In [3]:
STEADY_STATE_RESIDUAL_TOL = 1e-6
FALLBACK_MAX_STEPS        = 200000
FALLBACK_BURNIN_TIME      = 500000
ABS_TOL_DYDT              = 1e-10

### Residual Finder

This function assesses whether the system has reached a **steady state** by testing if every chemical's rate of change meets absolute or relative stability thresholds, set before. It is used as a helper function inside `reach_steady_state()`.

In [4]:
def verify_residual(r, rel_tol=STEADY_STATE_RESIDUAL_TOL, abs_tol=ABS_TOL_DYDT):
    rates_arr = r.getRatesOfChangeNamedArray() # Gets rates of change of each species
    species_ids = list(rates_arr.colnames) # Gets the species list
    dydt = np.array(rates_arr).flatten() # Array of rates of change of each species
    y = np.array([r[sid] for sid in species_ids]) # Array of concentration of each species
 
    abs_ok = np.abs(dydt) < abs_tol # Absolute Check
    rel_ratio = np.abs(dydt) / (np.abs(y) + 1e-12) # Relative Ratio Calculation
    rel_ok = rel_ratio < rel_tol # Relative Check
    converged = abs_ok | rel_ok # A species is considered at steady-state if it passes the absolute OR the relative check
 
    if np.all(converged): # If every single chemical in the entire model passed the test, the system is at steady state
        return 0.0
    return float(np.max(rel_ratio[~converged])) # If even one chemical failed, system is NOT at steady-state. Returns the chemical with the highest relative error.

### Steady-State Finder

The function encoded below finds the steady-state of each species in the model by using a three-phase approach. 
1. **Initial Burn-In (Phase 1)** - It runs a long (500,000 s) time-course simulation (using CVODE) to bring the system close to equilibrium.
2. **Newton Method (Phase 2)** - After the burn-in phase, it uses an algebraic root-finder (Newton's method) to instantly and precisely calculate the exact steady-state concentrations.
3. **Brute-Force Fallback (Phase 3)** - If the Newton method fails, it reverts to the time-stepper and simulates forward for another massive block of time (500,000 s), and we assume the concentrations at the end of the long run (totalling 1 million seconds) to be in steady-state.

*Note*: To check whether steady-state is reached, we use the `verify_residual()` function to confirm.

In [5]:
def reach_steady_state(r, warm_start_time=FALLBACK_BURNIN_TIME):
    # Setting the solver parameters
    integrator = r.getIntegrator()
    integrator.setValue('relative_tolerance', 1e-10)
    integrator.setValue('absolute_tolerance', 1e-12)
    integrator.maximum_num_steps = FALLBACK_MAX_STEPS

    # Initial Burn-in (Phase 1)
    try:
        r.simulate(0, warm_start_time, 2)
    except RuntimeError: # If solver throws an error here, terminate the function with a failure message
        return False, 'integration_fallback_exception', np.nan

    # Newton Method (Phase 2)
    try:
        ss_solver = r.getSteadyStateSolver()
        ss_solver.setValue('allow_approx', True)
        ss_solver.setValue('relative_tol', 1e-12)
        r.steadyState()
        residual = verify_residual(r) # Get residual
        if np.isfinite(residual) and residual < STEADY_STATE_RESIDUAL_TOL: # If residual is below set threshold, assume success (steady-state reached) and terminate the function
            return True, 'newton_warm', residual
    except RuntimeError:
        pass

    # Integration Fallback (Phase 3)
    try:
        r.simulate(0, FALLBACK_BURNIN_TIME, 2)
        residual = verify_residual(r) # Check residual again
        if np.isfinite(residual) and residual < STEADY_STATE_RESIDUAL_TOL: # If residual is below accepcted threshold, assume success and terminate function
            return True, 'integration_fallback', residual
        else:
            return False, 'no_steady_state', residual # If residual is large, inform steady-state has not been reached through the integration fallback
    except RuntimeError: # If solver throws an error here, integration fallback has failed to simulate for the set time
        return False, 'integration_fallback_exception', np.nan

### Experiment Controls and Time Points

The timings were directly obtained from **Materials and Methods** of **Uehara and Park (2002)**.
- t=0        : shared culture at 30 C, 15 Klett units (burn-in state)
- t=0..1800  : shift to 42 C (Theta gates ftsZ84 branches here)
- t=1800     : cefoxitin added to the two "+BL" branches
- t=1800..3600: 30 min induction period
- t=3600,5400,7200     : sampling points 

In [6]:
T_SHIFT = 0
T_DRUG = 1800 # Drug added at 30 min from start
T_SAMPLES = [3600, 5400, 7200] # 30 min, 60 min, 90 min (with respect to drug addition)
T_END = 7200 # time-course plotted over the full protocol (120 min)
N_POINTS = 721 # ~10 s resolution for smooth curves

# Global Optimization

This part of the script deals with the **optimization problem** i.e., finding the best parameter set.

## Free Parameters

The global optimisation procedure adjusts a predefined set of model parameters to improve agreement with the specified calibration targets. Each parameter is assigned a **lower bound**, an **upper bound**, and a **flag** indicating whether the optimisation should be performed on a **logarithmic scale**. These bounds constrain the search to the selected parameter ranges.

In [7]:
FREE_PARAMS_OPT = [
    ('V_max_BacA',     1e-3, 1e-1, True),
    ('V_max_DedA',     1e-3, 1e-1, True),
    ('v_de_novo',      1e-4, 5e-2, True),
    ('V_max_MraY',     1e-4, 5e-3, True),
    ('V_max_MurG',     1e-4, 1e-2, True),
    ('V_max_RodA',     1e-5, 1e-2, True),
    ('k_cat_PBP1a_GT', 0.05,  100,  True),
    ('k_cat_PBP5',     1e-4, 5.0,  True),
    ('k_cat_PBP6',     1e-4, 5.0,  True),
    ('V_max_Slt',      1e-3, 0.5, True),
    ('K_M_UndP',       0.002, 5.0,  True),
    ('K_I_MurF',       1e-3, 1e-1,  True)
]

## Calibration Targets and Sampling Times

The optimiser evaluates the model against **eight target ratios** representing **AmpC fold induction** and **relative metabolite concentrations** across the experimental conditions. An additional **soft target** constrains the absolute **UDP-MurNAc-pentapeptide** concentration in the first condition. The sampling times specify when the model outputs are extracted for comparison, measured in seconds from the start of simulation.

In [8]:
TARGET_RATIOS = {
    'fold_WT':      6.2, # AmpC fold-change at 30 min after cefoxitin addition in wild-type strains (AmpC_Condition_2 / AmpC_Condition_1), taken from Table 2 of Uehara and Park 2002
    'fold_ftsZ':    1.4, # AmpC fold-change at 30 min after cefoxitin addition in ftsZ84 strains (AmpC_Condition_4 / AmpC_Condition_3), taken from Table 2 of Uehara and Park 2002
    'fold_WT_60':   5.875, # AmpC fold-change at 60 min after cefoxitin addition in wild-type strains (AmpC_Condition_2 / AmpC_Condition_1), taken from Figure 1 of Uehara and Park 2002
    'fold_ftsZ_60': 1.4167, # AmpC fold-change at 60 min after cefoxitin addition in ftsZ84 strains (AmpC_Condition_4 / AmpC_Condition_3), taken from Figure 1 of Uehara and Park 2002
    'r3_AmpC1v3': 1.0, # (AmpC_Condition_1 / AmpC_Condition_3) = 1, assuming genotype does not affect AmpC baseline, taken from Figure 1 of Uehara and Park 2002
    'r4_UDP1v2':  3.7, # Ratio of repressor muropeptide in Condition 1 and Condition 2, taken from Uehara and Park Figure 3
    'r5_UDP3v1':  3.29, # Ratio of repressor muropeptide in Condition 3 and Condition 1, taken from Uehara and Park Figure 3
    'r6_UDP3v4':  1.68, # Ratio of repressor muropeptide in Condition 3 and Condition 4, taken from Uehara and Park Figure 3
}

UDP1_SOFT_TARGET = 0.2  # repressor muropeptide concentration in Condition 1, taken from Mengin-Lecreulx et al. 1982

# Sampling timepoints, in seconds from t=0 (t_drug=1800 -> 30/60 min post-drug)
T_SAMPLE_30 = 3600
T_SAMPLE_60 = 5400

## Loss Weights

The loss function combines discrepancies across the calibration targets into a single numerical objective. Each target is assigned a weight to control its contribution to the overall loss. **In this configuration, all eight ratio targets have equal weights, whereas the absolute UDP-MurNAc-pentapeptide target receives a smaller weight.**

In [9]:
LOSS_WEIGHTS = {
    'fold_WT':      1.0,
    'fold_ftsZ':    1.0,
    'fold_WT_60':   1.0,
    'fold_ftsZ_60': 1.0,
    'r3_AmpC1v3': 1.0,
    'r4_UDP1v2':  1.0,
    'r5_UDP3v1':  1.0,
    'r6_UDP3v4':  1.0,
    'UDP_1':      0.1, # lower weight, less priority
}

## Evaluating Candidate Parameter Set Function

The `evaluate_calibration()` function evaluates one candidate set of parameter values. It loads the mechanistic model, applies the proposed parameters, establishes a baseline steady state, and simulates the four experimental conditions. It then extracts AmpC and UDP-MurNAc-pentapeptide values at the specified sampling times and calculates the eight target ratios and the additional absolute concentration metric.

If the baseline calculation fails or the resulting measurements fail the function's validity checks, it returns None, allowing the loss function to penalize the candidate.

In [10]:
def evaluate_calibration(param_values):
    try:
        # Loading a fresh instance of the model
        r = te.loada(antimony_model)

        # Applying the candidate values to the corressponding parameters in the model
        for k, v in param_values.items():
            setattr(r, k, v)

        # Establishing the baseline under wild-type conditions without beta-lactam exposure
        r.ts_allele = 0
        r.drug_on = 0

        # Reaching steady state
        ok, method, residual = reach_steady_state(r)
        
        # Rejecting candidates for which baseline initialization (reaching steady state) fails
        if not ok:
            return None

        # Saving the baseline so all 4 conditions start from the same steady-state concentrations
        r.saveState('shared')

        # Storing AmpC measurements at 30 and 60 min after drug addition, repressor muropeptide at 30 min
        AmpC = {30: {}, 60: {}}
        UDP = {}

        # Condition labels
        conditions = {
            '1': (0, 0),
            '2': (0, 1),
            '3': (1, 0),
            '4': (1, 1)
        }
        
        for label, (ts, drug) in conditions.items():
            # Restoring the shared baseline state
            r.loadState('shared')

            # Configuring the configurables
            r.ts_allele = ts # FtsZ allele
            r.drug_on = drug # drug exposure
            r.t_shift = T_SHIFT
            r.t_drug = T_DRUG

            # Setting the maximum number of integration steps
            r.getIntegrator().maximum_num_steps = FALLBACK_MAX_STEPS
            
            # Simulating the complete time course (Sampling indices assume a 10 s output interval, T_END = 7200 s and N_POINTS = 721)
            result = r.simulate(0, T_END, N_POINTS)

            # Specifying the interested sampling indices (30 and 60 min observations after drug addition) 
            idx30 = int(round(T_SAMPLE_30 / 10))
            idx60 = int(round(T_SAMPLE_60 / 10))

            # Extracting AmpC concentrations at 30 and 60 min after drug addition
            AmpC[30][label] = result['AmpC'][idx30]
            AmpC[60][label] = result['AmpC'][idx60]

            # Extracting repressor muropeptide concentration at 30 min after drug addition
            UDP[label] = result['UDP_NAM_penta'][idx30]

        # Collecting all extracted measurements
        all_vals = ([AmpC[t][l] for t in (30, 60) for l in '1234']
                    + list(UDP.values()))

        # Rejecting results containing NaN or infinite values
        if not all(np.isfinite(all_vals)):
            return None

        # Ensuring the AmpC values used as ratio denominators are non-zero positive numbers
        if any(AmpC[t]['1'] <= 0 or AmpC[t]['3'] <= 0 for t in (30, 60)):
            return None

        # Ensuring the repressor muropeptide values used as ratio denominators are non-zero positive numbers
        if UDP['1'] <= 0 or UDP['4'] <= 0:
            return None

        # Calculating and returning the 8 calibration ratios along with the absolute repressor muropeptide concentration
        return {
            'fold_WT':      AmpC[30]['2'] / AmpC[30]['1'],
            'fold_ftsZ':    AmpC[30]['4'] / AmpC[30]['3'],
            'fold_WT_60':   AmpC[60]['2'] / AmpC[60]['1'],
            'fold_ftsZ_60': AmpC[60]['4'] / AmpC[60]['3'],
            'r3_AmpC1v3': AmpC[30]['1'] / AmpC[30]['3'],
            'r4_UDP1v2':  UDP['1'] / UDP['2'],
            'r5_UDP3v1':  UDP['3'] / UDP['1'],
            'r6_UDP3v4':  UDP['3'] / UDP['4'],
            'UDP_1':      UDP['1'],
        }
    # Treating runtime errors as failed candidate evaluations
    except RuntimeError:
        return None

## Calibration Loss Function

The `calibration_loss()` function converts a candidate parameter vector into its corresponding model parameter values, evaluates those values using `evaluate_calibration()`, and **calculates a weighted sum of squared relative deviations from the eight calibration targets**. A separate penalty accounts for the additional UDP-MurNAc-pentapeptide concentration target. Invalid candidate evaluations receive a large penalty.

This function supplies the numerical score that the global optimization algorithm attempts to minimize.

In [11]:
def calibration_loss(x, param_names, log_scale_flags):
    # Converting the candidate vector to model parameter values
    param_values = {
        name: (10 ** xi if is_log else xi)
        for name, xi, is_log in zip(param_names, x, log_scale_flags)
    }

    # Evaluating the candidate parameter set
    result = evaluate_calibration(param_values)

    # Penalizing invalid simulations with an artificial, high loss
    if result is None:
        return 1e6

    # Calculating the weighted loss for the eight target ratios
    loss = sum(
        LOSS_WEIGHTS[key] * ((result[key] / target) - 1) ** 2
        for key, target in TARGET_RATIOS.items()
    )

    # Adding the loss from the repressor muropeptide absolute concentration target
    loss += LOSS_WEIGHTS['UDP_1'] * ((result['UDP_1'] / UDP1_SOFT_TARGET) - 1) ** 2

    # Returning the loss value
    return loss

## Finding Optimal Parameters Function

The `find_optimal_parameters()` function uses SciPy's differential evolution algorithm to search the bounded parameter space for a low-loss solution. It converts the parameter bounds to logarithmic coordinates where required, defines an objective function that calls `calibration_loss()`, and runs the optimizer. Once the search finishes, it converts the best candidate back to the original parameter scale and evaluates its final metrics.

In [12]:
def find_optimal_parameters(maxiter=100, popsize=15, seed=42, workers=1):
    # Extracting parameter names and log flags
    param_names = [p[0] for p in FREE_PARAMS_OPT]
    log_flags = [p[3] for p in FREE_PARAMS_OPT]

    # Defining the search bounds for each parameter in log scale
    bounds = [
        (np.log10(lo), np.log10(hi)) if is_log else (lo, hi)
        for _, lo, hi, is_log in FREE_PARAMS_OPT
    ]

    # Defining the objective function that uses calibration_loss to return the loss (which differential evolution will minimize)
    def objective(x):
        return calibration_loss(x, param_names, log_flags)

    # Running and storing the results of differential evolution to find the optimal parameter values
    opt_result = differential_evolution(
        objective, 
        bounds,
        maxiter=maxiter, # Maximum number of generations
        popsize=popsize, # Population-size multiplier
        seed=seed,
        tol=1e-6, # Optmization convergence tolerance
        mutation=(0.5, 1.5), # Range of mutation factors
        recombination=0.7, # Crossover probability
        workers=workers, # Number of workers for objective evaluation
        polish=True, # Using local optimization to refine the best solution
        disp=True, # Displaying optimization progress
    )

    # Converting the optimal parameter values from logarithmic to linear scale
    best_params = {
        name: (10 ** xi if is_log else xi)
        for name, xi, is_log in zip(param_names, opt_result.x, log_flags)
    }

    # Re-evaluating the best parameter set to get the calibration metrics
    best_metrics = evaluate_calibration(best_params)

    # Returning the optimized parameter set, achieved metrics, and the actual optimization result from differential evolution
    return best_params, best_metrics, opt_result

# Main (Running Global Optimization, Saving and Reporting Results)

This cell executes the global optimization using the specified differential evolution settings, then reports the best-fit parameter values and compares the achieved metrics with their target values. It also displays the final loss and saves the optimized parameters to `optimal_parameters.json` for reuse in subsequent simulations without rerunning the optimization.

In [13]:
if __name__ == '__main__':
    # Running the global optimization and saving results
    print("\n---Global parameter optimization---")
    best_params, best_metrics, opt_result = find_optimal_parameters(maxiter=25, popsize=8, workers=1)
    
    # Displaying the optimal parameter values
    print("\nBest-fit parameters:")
    for k, v in best_params.items():
        print(f"  {k:18s} {v:.5e}")
    
    # Comparing the achieved values of the calibration ratios vs their targets
    print("\nAchieved vs. target ratios:")
    for key, target in TARGET_RATIOS.items():
        print(f"  {key:14s} achieved={best_metrics[key]:.4f}  target={target:.4f}")
    
    # Displaying the soft target (repressor muropeptide concentration in condition 1) 
    print(f"  {'UDP_1':14s} achieved={best_metrics['UDP_1']:.4f}  target~{UDP1_SOFT_TARGET}")
    
    # Reporting the final loss value
    print(f"\nFinal loss: {opt_result.fun:.6f}")
    
    # Saving the optimized parameter values to a JSON file
    with open('optimal_parameters.json', 'w') as f:
        json.dump(best_params, f, indent=2)


---Global parameter optimization---
differential_evolution step 1: f(x)= 1.8960643817674625
differential_evolution step 2: f(x)= 1.8960643817674625
differential_evolution step 3: f(x)= 1.8960643817674625
differential_evolution step 4: f(x)= 1.7673972512340477
differential_evolution step 5: f(x)= 1.7673972512340477
differential_evolution step 6: f(x)= 1.7160570642457285
differential_evolution step 7: f(x)= 1.7160570642457285
differential_evolution step 8: f(x)= 1.6658106454855135
differential_evolution step 9: f(x)= 1.6658106454855135
differential_evolution step 10: f(x)= 1.6658106454855135
differential_evolution step 11: f(x)= 1.6658106454855135
differential_evolution step 12: f(x)= 1.6658106454855135
differential_evolution step 13: f(x)= 1.6658106454855135
differential_evolution step 14: f(x)= 1.6658106454855135
differential_evolution step 15: f(x)= 1.6658106454855135
differential_evolution step 16: f(x)= 1.6658106454855135
differential_evolution step 17: f(x)= 1.6658106454855135
dif